# Lab 4 — Automatic Tag Generation for Ukrainian News

**Three approaches:**
1. Classical baseline (TF-IDF)
2. Zero-shot LLM (Qwen 2.5 3B via Unsloth)
3. Fine-tuned LLM with LoRA

**Hardware target:** RTX 3090 (24 GB VRAM)

## 0. Install dependencies

In [ ]:
# Run once
!pip install unsloth datasets scikit-learn sacrebleu tqdm transformers accelerate bitsandbytes --quiet

## 1. Load and prepare data

In [ ]:
from datasets import load_dataset
import random

random.seed(42)

print("Loading FIdo-AI/ua-news dataset...")
ds = load_dataset("FIdo-AI/ua-news", split="train")
print(f"Total records: {len(ds)}")
print(ds.features)
print(ds[0])

In [ ]:
# Filter records with non-empty tags and text
def has_tags(example):
    tags = example.get("tags", [])
    text = example.get("text", "") or ""
    title = example.get("title", "") or ""
    return tags and len(tags) > 0 and (len(text) + len(title)) > 50

ds_filtered = ds.filter(has_tags, num_proc=4)
print(f"Filtered records: {len(ds_filtered)}")

# Shuffle and split
ds_shuffled = ds_filtered.shuffle(seed=42)

TRAIN_SIZE = 5000
TEST_SIZE  = 1000

train_data = ds_shuffled.select(range(TRAIN_SIZE))
test_data  = ds_shuffled.select(range(TRAIN_SIZE, TRAIN_SIZE + TEST_SIZE))

print(f"Train: {len(train_data)}, Test: {len(test_data)}")

In [ ]:
def get_text(example, max_chars=500):
    """Combine title + body, truncate to max_chars."""
    title = example.get("title", "") or ""
    body  = example.get("text",  "") or ""
    combined = (title + "\n" + body).strip()
    return combined[:max_chars]

def get_tags(example):
    """Return list of tag strings."""
    tags = example.get("tags", [])
    if isinstance(tags, str):
        tags = [t.strip() for t in tags.split(",") if t.strip()]
    return [str(t).strip().lower() for t in tags if t]

# Quick EDA
import numpy as np
tag_counts = [len(get_tags(x)) for x in test_data]
print(f"Tags per article — mean: {np.mean(tag_counts):.1f}, median: {np.median(tag_counts):.1f}, max: {max(tag_counts)}")
print("Example article:")
print(get_text(test_data[0]))
print("Tags:", get_tags(test_data[0]))

## 2. Evaluation utilities

In [ ]:
from sacrebleu.metrics import CHRF

chrf_metric = CHRF(word_order=2)  # chrF++

def compute_f1(predicted_tags: list, true_tags: list):
    """Token-level F1 over tag sets."""
    pred_set = set(t.lower().strip() for t in predicted_tags if t.strip())
    true_set = set(t.lower().strip() for t in true_tags if t.strip())
    if not pred_set and not true_set:
        return 1.0, 1.0, 1.0
    if not pred_set or not true_set:
        return 0.0, 0.0, 0.0
    tp = len(pred_set & true_set)
    precision = tp / len(pred_set)
    recall    = tp / len(true_set)
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    return precision, recall, f1

def compute_chrf(predicted_tags: list, true_tags: list):
    """chrF++ on joined tag strings."""
    pred_str = " | ".join(predicted_tags)
    true_str = " | ".join(true_tags)
    if not pred_str or not true_str:
        return 0.0
    score = chrf_metric.sentence_score(pred_str, [true_str])
    return score.score

def evaluate(all_predicted, all_true):
    """Compute macro-averaged F1 and chrF++ over lists of tag lists."""
    precisions, recalls, f1s, chrfs = [], [], [], []
    for pred, true in zip(all_predicted, all_true):
        p, r, f = compute_f1(pred, true)
        precisions.append(p)
        recalls.append(r)
        f1s.append(f)
        chrfs.append(compute_chrf(pred, true))
    return {
        "precision": float(np.mean(precisions)),
        "recall":    float(np.mean(recalls)),
        "f1":        float(np.mean(f1s)),
        "chrf++":    float(np.mean(chrfs)),
    }

print("Evaluation utilities ready.")

## Part 1 — Classical Baseline (TF-IDF)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

# Build TF-IDF on training corpus
train_texts = [get_text(x) for x in train_data]
test_texts  = [get_text(x) for x in test_data]
test_true   = [get_tags(x) for x in test_data]

vectorizer = TfidfVectorizer(
    max_features=50000,
    ngram_range=(1, 2),
    min_df=2,
    sublinear_tf=True,
    token_pattern=r"(?u)\b[а-яА-ЯіІїЇєЄёЁa-zA-Z][а-яА-ЯіІїЇєЄёЁa-zA-Z\-']{2,}\b",
)

print("Fitting TF-IDF on training corpus...")
vectorizer.fit(train_texts)
print(f"Vocabulary size: {len(vectorizer.vocabulary_)}")

In [ ]:
def tfidf_predict(text, vectorizer, top_n=7):
    """Return top-N keywords by TF-IDF score for a single text."""
    vec = vectorizer.transform([text])
    feature_names = vectorizer.get_feature_names_out()
    scores = vec.toarray()[0]
    top_indices = scores.argsort()[::-1][:top_n]
    return [feature_names[i] for i in top_indices if scores[i] > 0]

print("Running TF-IDF on test set...")
tfidf_predictions = [tfidf_predict(t, vectorizer) for t in test_texts]

# Show a few examples
for i in range(3):
    print(f"\n--- Example {i+1} ---")
    print("Text snippet:", test_texts[i][:200])
    print("TF-IDF tags: ", tfidf_predictions[i])
    print("True tags:   ", test_true[i])

In [ ]:
results_tfidf = evaluate(tfidf_predictions, test_true)
print("=== TF-IDF Baseline ===")
for k, v in results_tfidf.items():
    print(f"  {k:12s}: {v:.4f}")

## Part 2 — Zero-shot LLM

In [ ]:
from unsloth import FastLanguageModel
import torch

MODEL_NAME = "unsloth/Qwen2.5-3B-Instruct"
MAX_SEQ_LEN = 1024

print(f"Loading model: {MODEL_NAME}")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LEN,
    dtype=torch.bfloat16,
    load_in_4bit=False,   # 3090 has 24 GB — no need for 4bit
)
FastLanguageModel.for_inference(model)
print("Model loaded!")

In [ ]:
SYSTEM_PROMPT = (
    "Ти — асистент для аналізу українських новин. "
    "Твоє завдання: згенерувати від 5 до 10 інформативних тегів для статті. "
    "Теги мають бути короткими (1-3 слова), розділеними символом '|'. "
    "Відповідай ТІЛЬКИ тегами, без пояснень."
)

def build_prompt(text):
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": f"Стаття:\n{text}\n\nТеги:"},
    ]

def parse_tags(raw_output: str):
    """Parse model output — split by | or ,"""
    # Remove any accidental extra text after newline
    raw_output = raw_output.split("\n")[0]
    for sep in ["|", ",", ";"]:
        if sep in raw_output:
            tags = [t.strip().lower() for t in raw_output.split(sep) if t.strip()]
            return tags[:15]
    return [raw_output.strip().lower()] if raw_output.strip() else []

def predict_tags_llm(text, model, tokenizer, max_new_tokens=128):
    messages = build_prompt(text)
    input_ids = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt",
    ).to(model.device)
    with torch.inference_mode():
        output_ids = model.generate(
            input_ids,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=1.0,
            pad_token_id=tokenizer.eos_token_id,
        )
    generated = output_ids[0][input_ids.shape[1]:]
    raw = tokenizer.decode(generated, skip_special_tokens=True)
    return parse_tags(raw)

# Quick test
sample = test_texts[0]
pred = predict_tags_llm(sample, model, tokenizer)
print("Sample prediction:", pred)
print("True tags:        ", test_true[0])

In [ ]:
from tqdm.auto import tqdm

print("Running zero-shot inference on test set (this may take ~15-30 min)...")
zeroshot_predictions = []
for text in tqdm(test_texts):
    tags = predict_tags_llm(text, model, tokenizer)
    zeroshot_predictions.append(tags)

results_zeroshot = evaluate(zeroshot_predictions, test_true)
print("\n=== Zero-shot LLM ===")
for k, v in results_zeroshot.items():
    print(f"  {k:12s}: {v:.4f}")

## Part 3 — Fine-tuned LLM with LoRA

In [ ]:
# Add LoRA adapters to the already-loaded model
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)
print(model.print_trainable_parameters())

In [ ]:
from datasets import Dataset

def format_example(example):
    """Format one training example as a chat with assistant answer."""
    text = get_text(example)
    tags = get_tags(example)
    tags_str = " | ".join(tags)
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": f"Стаття:\n{text}\n\nТеги:"},
        {"role": "assistant", "content": tags_str},
    ]
    formatted = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": formatted}

print("Formatting training data...")
train_formatted = [format_example(x) for x in tqdm(train_data)]
train_dataset = Dataset.from_list(train_formatted)

print("Example formatted record:")
print(train_dataset[0]["text"][:600])

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LEN,
    dataset_num_proc=2,
    packing=True,  # packing speeds up training significantly
    args=TrainingArguments(
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,   # effective batch = 16
        warmup_steps=50,
        num_train_epochs=2,
        learning_rate=2e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=20,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=42,
        output_dir="./lora_checkpoints",
        report_to="none",
        save_strategy="no",
    ),
)

print("Starting LoRA fine-tuning...")
trainer_stats = trainer.train()
print("Training complete!")
print(f"Training loss: {trainer_stats.training_loss:.4f}")

In [ ]:
# Switch model to inference mode
FastLanguageModel.for_inference(model)

print("Running fine-tuned inference on test set...")
finetuned_predictions = []
for text in tqdm(test_texts):
    tags = predict_tags_llm(text, model, tokenizer)
    finetuned_predictions.append(tags)

results_finetuned = evaluate(finetuned_predictions, test_true)
print("\n=== Fine-tuned LLM ===")
for k, v in results_finetuned.items():
    print(f"  {k:12s}: {v:.4f}")

## 4. Results comparison

In [ ]:
import pandas as pd

comparison = pd.DataFrame({
    "TF-IDF":     results_tfidf,
    "Zero-shot":  results_zeroshot,
    "Fine-tuned": results_finetuned,
}).T

print("\n" + "="*55)
print("       FINAL RESULTS COMPARISON")
print("="*55)
print(comparison.to_string(float_format="{:.4f}".format))
print("="*55)

In [ ]:
# Qualitative examples
print("\n--- Qualitative comparison on 5 test examples ---")
for i in range(5):
    print(f"\n{'='*60}")
    print(f"ARTICLE ({i+1}): {test_texts[i][:250]}")
    print(f"TRUE:        {test_true[i]}")
    print(f"TF-IDF:      {tfidf_predictions[i]}")
    print(f"Zero-shot:   {zeroshot_predictions[i]}")
    print(f"Fine-tuned:  {finetuned_predictions[i]}")

## 5. Save the LoRA adapter

In [ ]:
model.save_pretrained("./ua_news_lora")
tokenizer.save_pretrained("./ua_news_lora")
print("LoRA adapter saved to ./ua_news_lora")

# Optional: save merged model in GGUF for local inference with llama.cpp / Ollama
# model.save_pretrained_gguf("ua_news_q8", tokenizer, quantization_method="q8_0")

## 6. Analysis & conclusions

**TF-IDF** — простий статистичний підхід. Добре знаходить слова, що часто зустрічаються в конкретній статті, але рідкісні в корпусі. Не розуміє семантики, не генерує теги, яких немає в тексті.

**Zero-shot LLM** — модель без донавчання. Генерує семантично осмислені теги, може узагальнювати і формулювати теми, яких не було в тексті буквально. Але не знає специфічного формату тегів українського новинного агентства.

**Fine-tuned LLM** — після LoRA донавчання модель адаптується до стилю та розподілу тегів датасету. Очікувано показує найвищий F1 і chrF++, оскільки навчилась відтворювати саме той формат і словниковий запас тегів, які використовуються в датасеті.

**Висновок:** Fine-tuning справді дає приріст якості, коли є достатньо даних зі специфічним розподілом відповідей (формат, мова, стиль). Zero-shot LLM є сильним baseline'ом для задач, де формат відповіді більш відкритий.